In [28]:
# ============================================================
# CREDIT RISK ANALYTICS SYSTEM
# FEATURE ENGINEERING
# ============================================================

import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [29]:
# ============================================================
# 1. LOAD ORIGINAL BORROWER DATASET
# ============================================================

DATA_PATH = "../data/synthetic/borrower_credit_data.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("ORIGINAL DATASET LOADED")
print("=" * 60)

print("\nShape:", df.shape)

print("\nFirst 5 Records:")
display(df.head())

ORIGINAL DATASET LOADED

Shape: (10000, 28)

First 5 Records:


,borrower_id,bank,age,employment_type,employment_length_years,annual_income,credit_score,loan_amount,loan_term_months,interest_rate,existing_debt,existing_loans,previous_delinquencies,previous_defaults,credit_history_years,open_credit_accounts,dependents,home_ownership,loan_purpose,monthly_income,monthly_emi,debt_to_income_ratio,loan_to_income_ratio,bank_gross_npas,bank_gross_advances,bank_gross_npa_ratio,risk_category,default_flag
0,BRW000001,STATE BANK OF INDIA,52,Business Owner,7.2000,"495,999.0000",602,"1,037,176.0000",120,9.5000,"110,589.0000",0,1,0,28,1,2,Own,Medical,"41,333.2500","13,420.8000",0.5500,2.0900,"76,880.2000","4,220,702.7400",1.8200,Medium,0
1,BRW000002,ICICI BANK LIMITED,56,Salaried,8.8000,"346,767.0000",643,"2,020,637.0000",60,13.0800,"66,739.0000",1,0,0,34,2,4,Own,Vehicle,"28,897.2500","46,058.4900",1.7900,5.8300,"23,523.1000","1,359,789.8900",1.7300,High,0
2,BRW000003,DCB BANK LIMITED,57,Salaried,7.1000,"710,300.0000",688,"1,144,821.0000",60,15.2100,"511,044.0000",2,0,0,39,2,0,Rent,Education,"59,191.6700","27,361.5700",1.1800,1.6100,"1,554.4300","52,047.5300",2.9900,Medium,0
3,BRW000004,AXIS BANK LIMITED,34,Salaried,8.1000,"769,394.0000",843,"1,547,261.0000",36,12.2600,"205,078.0000",2,1,0,11,1,2,Own,Education,"64,116.1700","51,583.5600",1.0700,2.0100,"13,961.1900","1,051,087.0600",1.3300,Low,0
4,BRW000005,CANARA BANK,47,Self-Employed,2.2000,"180,000.0000",673,"569,667.0000",120,8.8600,"187,834.0000",0,0,0,25,7,4,Mortgage,Vehicle,"15,000.0000","7,173.2100",1.5200,3.1600,"31,530.0300","1,073,331.7400",2.9400,Medium,0


In [30]:
# ============================================================
# 2. TARGET AND POTENTIAL LEAKAGE CHECK
# ============================================================

TARGET = "default_flag"

print("=" * 60)
print("TARGET / LEAKAGE CHECK")
print("=" * 60)

print("\nTarget Variable:")
print(TARGET)

print("\nTarget Distribution:")
print(df[TARGET].value_counts())

print("\nTarget Rate:")
print(f"{df[TARGET].mean() * 100:.2f}%")

print("\nPotential Leakage / Identifier Columns:")

potential_leakage_columns = [
    "borrower_id",
    "default_flag",
    "risk_category"
]

for column in potential_leakage_columns:
    print(f"- {column}")

TARGET / LEAKAGE CHECK

Target Variable:
default_flag

Target Distribution:
default_flag
0    9300
1     700
Name: count, dtype: int64

Target Rate:
7.00%

Potential Leakage / Identifier Columns:
- borrower_id
- default_flag
- risk_category


In [31]:
# ============================================================
# 3. DEFINE FEATURE GROUPS
# ============================================================

numerical_features = [
    "age",
    "employment_length_years",
    "annual_income",
    "credit_score",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "existing_debt",
    "existing_loans",
    "previous_delinquencies",
    "previous_defaults",
    "credit_history_years",
    "open_credit_accounts",
    "dependents",
    "monthly_income",
    "monthly_emi",
    "debt_to_income_ratio",
    "loan_to_income_ratio",
    "bank_gross_npa_ratio"
]

categorical_features = [
    "bank",
    "employment_type",
    "home_ownership",
    "loan_purpose"
]

print("=" * 60)
print("FEATURE GROUPS")
print("=" * 60)

print("\nNumerical Features:", len(numerical_features))

for feature in numerical_features:
    print("-", feature)

print("\nCategorical Features:", len(categorical_features))

for feature in categorical_features:
    print("-", feature)

FEATURE GROUPS

Numerical Features: 19
- age
- employment_length_years
- annual_income
- credit_score
- loan_amount
- loan_term_months
- interest_rate
- existing_debt
- existing_loans
- previous_delinquencies
- previous_defaults
- credit_history_years
- open_credit_accounts
- dependents
- monthly_income
- monthly_emi
- debt_to_income_ratio
- loan_to_income_ratio
- bank_gross_npa_ratio

Categorical Features: 4
- bank
- employment_type
- home_ownership
- loan_purpose


In [32]:
# ============================================================
# 4. FEATURE VALIDATION
# ============================================================

required_features = (
    numerical_features
    + categorical_features
    + [TARGET]
)

missing_features = [
    feature
    for feature in required_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "Missing required features:\n"
        + "\n".join(missing_features)
    )

print("✓ All required features are present.")
print("✓ Feature validation successful.")

✓ All required features are present.
✓ Feature validation successful.


In [33]:
# ============================================================
# 5. MISSING VALUE CHECK
# ============================================================

print("=" * 60)
print("MISSING VALUE CHECK")
print("=" * 60)

missing = df[required_features].isnull().sum()

missing = missing[missing > 0]

if missing.empty:
    print("✓ No missing values found.")
else:
    print("Missing values detected:")
    print(missing)

MISSING VALUE CHECK
✓ No missing values found.


In [34]:
# ============================================================
# 6. EMI-TO-INCOME RATIO
# ============================================================

df["emi_to_income_ratio"] = (
    df["monthly_emi"] /
    df["monthly_income"]
)

df["emi_to_income_ratio"] = (
    df["emi_to_income_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .clip(0, 3)
    .round(4)
)

print("EMI-to-Income Ratio created successfully.")

print("\nSummary:")
print(
    df["emi_to_income_ratio"]
    .describe()
)

EMI-to-Income Ratio created successfully.

Summary:
count   10,000.0000
mean         0.5468
std          0.5849
min          0.0089
25%          0.1769
50%          0.3440
75%          0.6746
max          3.0000
Name: emi_to_income_ratio, dtype: float64


In [35]:
# ============================================================
# 7. TOTAL DEBT EXPOSURE
# ============================================================

df["total_debt_exposure"] = (
    df["existing_debt"] +
    df["loan_amount"]
)

print("Total Debt Exposure created.")

print("\nSummary:")
print(
    df["total_debt_exposure"]
    .describe()
)

Total Debt Exposure created.

Summary:
count      10,000.0000
mean    1,427,088.2987
std       812,206.2662
min       105,601.0000
25%       854,566.5000
50%     1,249,820.0000
75%     1,807,461.2500
max     6,733,163.0000
Name: total_debt_exposure, dtype: float64


In [36]:
# ============================================================
# 8. CREDIT HISTORY AGE RATIO
# ============================================================

df["credit_history_age_ratio"] = (
    df["credit_history_years"] /
    df["age"]
)

df["credit_history_age_ratio"] = (
    df["credit_history_age_ratio"]
    .clip(0, 1)
    .round(4)
)

print("Credit History Age Ratio created.")

print(
    df["credit_history_age_ratio"]
    .describe()
)

Credit History Age Ratio created.
count   10,000.0000
mean         0.4338
std          0.1803
min          0.0400
25%          0.3103
50%          0.4762
75%          0.5833
max          0.6897
Name: credit_history_age_ratio, dtype: float64


In [37]:
# ============================================================
# 9. DELINQUENCY INDICATOR
# ============================================================

df["has_previous_delinquency"] = (
    df["previous_delinquencies"] > 0
).astype(int)

print("Previous Delinquency Indicator created.")

print(
    df["has_previous_delinquency"]
    .value_counts()
)

Previous Delinquency Indicator created.
has_previous_delinquency
0    6982
1    3018
Name: count, dtype: int64


In [38]:
# ============================================================
# 10. PREVIOUS DEFAULT INDICATOR
# ============================================================

df["has_previous_default"] = (
    df["previous_defaults"] > 0
).astype(int)

print("Previous Default Indicator created.")

print(
    df["has_previous_default"]
    .value_counts()
)

Previous Default Indicator created.
has_previous_default
0    9200
1     800
Name: count, dtype: int64


In [39]:
# ============================================================
# 11. ENGINEERED FEATURE CORRELATION
# ============================================================

engineered_features = [
    "emi_to_income_ratio",
    "total_debt_exposure",
    "credit_history_age_ratio",
    "has_previous_delinquency",
    "has_previous_default"
]

correlation_check = (
    df[engineered_features + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values(
        key=abs,
        ascending=False
    )
)

print("=" * 60)
print("ENGINEERED FEATURE CORRELATION WITH DEFAULT")
print("=" * 60)

correlation_check

ENGINEERED FEATURE CORRELATION WITH DEFAULT


emi_to_income_ratio         0.2583
has_previous_default        0.1560
total_debt_exposure         0.1526
has_previous_delinquency    0.1133
credit_history_age_ratio   -0.0002
Name: default_flag, dtype: float64

In [40]:
# ============================================================
# 12. FINAL MODEL FEATURES
# ============================================================

engineered_numerical_features = [
    "emi_to_income_ratio",
    "total_debt_exposure",
    "credit_history_age_ratio",
    "has_previous_delinquency",
    "has_previous_default"
]

final_numerical_features = (
    numerical_features +
    engineered_numerical_features
)

final_categorical_features = categorical_features.copy()

final_features = (
    final_numerical_features +
    final_categorical_features
)

print("=" * 60)
print("FINAL MODEL FEATURE SET")
print("=" * 60)

print("\nNumerical Features:", len(final_numerical_features))

for feature in final_numerical_features:
    print("-", feature)

print("\nCategorical Features:", len(final_categorical_features))

for feature in final_categorical_features:
    print("-", feature)

print("\nTotal Features:", len(final_features))

FINAL MODEL FEATURE SET

Numerical Features: 24
- age
- employment_length_years
- annual_income
- credit_score
- loan_amount
- loan_term_months
- interest_rate
- existing_debt
- existing_loans
- previous_delinquencies
- previous_defaults
- credit_history_years
- open_credit_accounts
- dependents
- monthly_income
- monthly_emi
- debt_to_income_ratio
- loan_to_income_ratio
- bank_gross_npa_ratio
- emi_to_income_ratio
- total_debt_exposure
- credit_history_age_ratio
- has_previous_delinquency
- has_previous_default

Categorical Features: 4
- bank
- employment_type
- home_ownership
- loan_purpose

Total Features: 28


In [41]:
# ============================================================
# 13. TRAIN / TEST SPLIT
# ============================================================

X = df[final_features].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("\nTraining Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

print("\nTraining Default Rate:")
print(f"{y_train.mean() * 100:.2f}%")

print("\nTesting Default Rate:")
print(f"{y_test.mean() * 100:.2f}%")

TRAIN / TEST SPLIT

Training Features: (8000, 28)
Testing Features: (2000, 28)

Training Target Distribution:
default_flag
0    7440
1     560
Name: count, dtype: int64

Testing Target Distribution:
default_flag
0    1860
1     140
Name: count, dtype: int64

Training Default Rate:
7.00%

Testing Default Rate:
7.00%


In [42]:
# ============================================================
# 14. FINAL LEAKAGE CHECK
# ============================================================

forbidden_features = [
    "borrower_id",
    "default_flag",
    "risk_category",
    "credit_score_band",
    "dti_band"
]

leakage_found = [
    feature
    for feature in forbidden_features
    if feature in final_features
]

print("=" * 60)
print("FINAL LEAKAGE CHECK")
print("=" * 60)

if leakage_found:
    raise ValueError(
        "Potential leakage detected:\n"
        + "\n".join(leakage_found)
    )

print("✓ No target/identifier/risk-category leakage detected.")

FINAL LEAKAGE CHECK
✓ No target/identifier/risk-category leakage detected.
